In [2]:


import polars as pl

flights = (pl.scan_parquet("../data/flights.parquet")
           .with_columns(
    pl.col('YEAR').cast(pl.String).cast(pl.Int32),
    pl.col('MONTH').cast(pl.String).cast(pl.Int32),
    pl.col('DAY').cast(pl.String).cast(pl.Int32),
    pl.col('DAY_OF_WEEK').cast(pl.String).cast(pl.Int32),
))

airlines = pl.scan_parquet("../data/airlines.parquet")
airports = pl.scan_parquet("../data/airports.parquet")

# Predicting delays

## Data preparation

Only not canceled and not diverted flights will be considered in models.

In [3]:
flights_delays = flights.filter(pl.col('CANCELLED') == 0, pl.col('DIVERTED') == 0)

Checking nulls

In [4]:
(flights_delays
 .null_count()
 .unpivot()
 .filter(pl.col('value') > 0)
 .select(pl.len().alias("No of nulls"))
 .collect())

No of nulls
u32
0


Column analysis

In [5]:
flights_delays.collect_schema().names()

['YEAR',
 'MONTH',
 'DAY',
 'DAY_OF_WEEK',
 'AIRLINE',
 'FLIGHT_NUMBER',
 'TAIL_NUMBER',
 'ORIGIN_AIRPORT',
 'DESTINATION_AIRPORT',
 'SCHEDULED_DEPARTURE',
 'DEPARTURE_TIME',
 'DEPARTURE_DELAY',
 'TAXI_OUT',
 'WHEELS_OFF',
 'SCHEDULED_TIME',
 'ELAPSED_TIME',
 'AIR_TIME',
 'DISTANCE',
 'WHEELS_ON',
 'TAXI_IN',
 'SCHEDULED_ARRIVAL',
 'ARRIVAL_TIME',
 'ARRIVAL_DELAY',
 'DIVERTED',
 'CANCELLED',
 'CANCELLATION_REASON',
 'AIR_SYSTEM_DELAY',
 'SECURITY_DELAY',
 'AIRLINE_DELAY',
 'LATE_AIRCRAFT_DELAY',
 'WEATHER_DELAY']

There are some columns that won't be needed in the model:

  - Too many different values
     - FLIGHT_NUMBER – to many text unique values
     - TAIL_NUMBER – to many text unique values, but this column will be used for feature engineering
  - Future values that shouldn't be visible for the model
    - DEPARTURE_TIME
    - DEPARTURE_DELAY – temporarily used for feature engineering
    - TAXI_OUT
    - WHEELS_OFF
    - ELAPSED_TIME
    - AIR_TIME
    - WHEELS_ON
    - TAXI_IN
    - ARRIVAL_TIME
  - Values that are not important for these predictions
    - DIVERTED
    - CANCELLED
    - CANCELLATION_REASON
    - AIR_SYSTEM_DELAY
    - SECURITY_DELAY
    - AIRLINE_DELAY
    - LATE_AIRCRAFT_DELAY
    - WEATHER_DELAY
  - Values that will be converted to another in feature engineering
    - SCHEDULED_DEPARTURE
    - SCHEDULED_ARRIVAL

And these columns will be dropped.

In [6]:
flights_delays_reduced = flights_delays.drop(
    'FLIGHT_NUMBER',
    'DEPARTURE_TIME',
    'TAXI_OUT',
    'WHEELS_OFF',
    'ELAPSED_TIME',
    'AIR_TIME',
    'WHEELS_ON',
    'TAXI_IN',
    'ARRIVAL_TIME',
    'DIVERTED',
    'CANCELLED',
    'CANCELLATION_REASON',
    'AIR_SYSTEM_DELAY',
    'SECURITY_DELAY',
    'AIRLINE_DELAY',
    'LATE_AIRCRAFT_DELAY',
    'WEATHER_DELAY'
)
flights_delays_reduced.collect_schema().names()

['YEAR',
 'MONTH',
 'DAY',
 'DAY_OF_WEEK',
 'AIRLINE',
 'TAIL_NUMBER',
 'ORIGIN_AIRPORT',
 'DESTINATION_AIRPORT',
 'SCHEDULED_DEPARTURE',
 'DEPARTURE_DELAY',
 'SCHEDULED_TIME',
 'DISTANCE',
 'SCHEDULED_ARRIVAL',
 'ARRIVAL_DELAY']

## Basic statistics of features

In [7]:
flights_delays_reduced.collect().describe()

statistic,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_DELAY
str,f64,f64,f64,f64,str,str,str,str,str,f64,f64,f64,str,f64
"""count""",5.714008e6,5.714008e6,5.714008e6,5.714008e6,"""5714008""","""5714008""","""5714008""","""5714008""","""5714008""",5.714008e6,5.714008e6,5.714008e6,"""5714008""",5.714008e6
"""null_count""",0.0,0.0,0.0,0.0,"""0""","""0""","""0""","""0""","""0""",0.0,0.0,0.0,"""0""",0.0
"""mean""",2015.0,6.547799,15.707591,3.932643,null,null,null,null,"""13:28:01.186673""",9.294842,141.893974,824.456903,"""15:07:26.994575""",4.407057
"""std""",0.0,3.397421,8.774394,1.985967,null,null,null,null,null,36.889724,75.313998,608.66199,null,39.271297
"""min""",2015.0,1.0,1.0,1.0,null,null,null,null,"""00:01:00""",-82.0,18.0,31.0,"""00:00:00""",-87.0
"""25%""",2015.0,4.0,8.0,2.0,null,null,null,null,"""09:16:00""",-5.0,85.0,373.0,"""11:10:00""",-13.0
"""50%""",2015.0,7.0,16.0,4.0,null,null,null,null,"""13:25:00""",-2.0,123.0,650.0,"""15:20:00""",-5.0
"""75%""",2015.0,9.0,23.0,6.0,null,null,null,null,"""17:30:00""",7.0,174.0,1065.0,"""19:17:00""",8.0
"""max""",2015.0,12.0,31.0,7.0,null,null,null,null,"""23:59:00""",1988.0,718.0,4983.0,"""23:59:00""",1971.0


## Non logical values check

In [8]:
print(flights_delays_reduced.filter((pl.col("YEAR").cast(pl.String).cast(pl.Int32) < 2015) |
                                    (pl.col("YEAR").cast(pl.String).cast(pl.Int32) > 2015) |
                                    (pl.col("MONTH").cast(pl.String).cast(pl.Int32) < 1) |
                                    (pl.col("MONTH").cast(pl.String).cast(pl.Int32) > 12) |
                                    (pl.col("DAY").cast(pl.String).cast(pl.Int32) < 1) |
                                    (pl.col("DAY").cast(pl.String).cast(pl.Int32) > 31) |
                                    (pl.col("DAY_OF_WEEK").cast(pl.String).cast(pl.Int32) < 1) |
                                    (pl.col("DAY_OF_WEEK").cast(pl.String).cast(pl.Int32) > 7) |
                                    (pl.col("SCHEDULED_DEPARTURE") < pl.time(0, 0, 0)) |
                                    (pl.col("SCHEDULED_DEPARTURE") > pl.time(23, 59, 0))
                                    ).collect())

shape: (0, 14)
┌──────┬───────┬─────┬─────────────┬───┬────────────────┬──────────┬───────────────┬───────────────┐
│ YEAR ┆ MONTH ┆ DAY ┆ DAY_OF_WEEK ┆ … ┆ SCHEDULED_TIME ┆ DISTANCE ┆ SCHEDULED_ARR ┆ ARRIVAL_DELAY │
│ ---  ┆ ---   ┆ --- ┆ ---         ┆   ┆ ---            ┆ ---      ┆ IVAL          ┆ ---           │
│ i32  ┆ i32   ┆ i32 ┆ i32         ┆   ┆ f64            ┆ i64      ┆ ---           ┆ f64           │
│      ┆       ┆     ┆             ┆   ┆                ┆          ┆ time          ┆               │
╞══════╪═══════╪═════╪═════════════╪═══╪════════════════╪══════════╪═══════════════╪═══════════════╡
└──────┴───────┴─────┴─────────────┴───┴────────────────┴──────────┴───────────────┴───────────────┘


## Rate of dominant value occurence

In [9]:
import polars as pl


def mode_stats(df: pl.LazyFrame, cols: list[str]) -> pl.DataFrame:
    total = df.select(pl.len()).collect().item()

    results = []
    for col in cols:
        mode_row = (
            df
            .select(pl.col(col))
            .filter(pl.col(col).is_not_null())
            .group_by(col)
            .agg(pl.len().alias("count"))
            .sort("count", descending=True)
            .head(1)
            .collect()
        )

        if mode_row.shape[0] > 0:
            mode_val = mode_row[col][0]
            mode_count = mode_row["count"][0]
            mode_pct = round(mode_count / total * 100, 2)
        else:
            mode_val, mode_count, mode_pct = None, 0, 0.0

        results.append({
            "column": col,
            "mode": str(mode_val),
            "count": mode_count,
            "pct": mode_pct,
        })

    return pl.DataFrame(results)


mode_stats(flights_delays_reduced, flights_delays_reduced.collect_schema().names())

column,mode,count,pct
str,str,i64,f64
"""YEAR""","""2015""",5714008,100.0
"""MONTH""","""7""",514384,9.0
"""DAY""","""13""",193233,3.38
"""DAY_OF_WEEK""","""4""",857886,15.01
"""AIRLINE""","""WN""",1242403,21.74
…,…,…,…
"""DEPARTURE_DELAY""","""-3.0""",454411,7.95
"""SCHEDULED_TIME""","""85.0""",112914,1.98
"""DISTANCE""","""337""",48731,0.85


There is only one year in dataset, so this column must be dropped, but for now it will stay and be used for feature engineering

In [10]:
#flights_delays_reduced = flights_delays_reduced.drop('YEAR')

## Categorical values occurrence check

In [11]:

def categorical_values_occurrence(data_frame):
    CATEGORICAL_COLS = data_frame.select(pl.selectors.by_dtype(pl.Categorical)).collect_schema().names()

    total = data_frame.select(pl.len()).collect().item()
    results = []
    for col in CATEGORICAL_COLS:
        results.append(data_frame
                       .group_by(col)
                       .agg(pl.len().alias('count'))
                       .with_columns((pl.col('count') / total).alias('rate'))
                       .sort('rate')
                       .collect())
        print(results[-1]
              )
    return results


analysis_results = categorical_values_occurrence(flights_delays_reduced)

shape: (14, 3)
┌─────────┬─────────┬──────────┐
│ AIRLINE ┆ count   ┆ rate     │
│ ---     ┆ ---     ┆ ---      │
│ cat     ┆ u32     ┆ f64      │
╞═════════╪═════════╪══════════╡
│ VX      ┆ 61248   ┆ 0.010719 │
│ HA      ┆ 76041   ┆ 0.013308 │
│ F9      ┆ 90090   ┆ 0.015767 │
│ NK      ┆ 115193  ┆ 0.02016  │
│ AS      ┆ 171439  ┆ 0.030003 │
│ …       ┆ …       ┆ …        │
│ EV      ┆ 554752  ┆ 0.097086 │
│ OO      ┆ 576814  ┆ 0.100947 │
│ AA      ┆ 712935  ┆ 0.12477  │
│ DL      ┆ 870275  ┆ 0.152306 │
│ WN      ┆ 1242403 ┆ 0.217431 │
└─────────┴─────────┴──────────┘
shape: (4_896, 3)
┌─────────────┬───────┬───────────┐
│ TAIL_NUMBER ┆ count ┆ rate      │
│ ---         ┆ ---   ┆ ---       │
│ cat         ┆ u32   ┆ f64       │
╞═════════════╪═══════╪═══════════╡
│ N175UA      ┆ 1     ┆ 1.7501e-7 │
│ N180UA      ┆ 1     ┆ 1.7501e-7 │
│ N121UA      ┆ 1     ┆ 1.7501e-7 │
│ N7LEAA      ┆ 1     ┆ 1.7501e-7 │
│ N840MH      ┆ 1     ┆ 1.7501e-7 │
│ …           ┆ …     ┆ …         │
│ N478HA  

There is only a problem with Airports. There are some with without IATA code, instead there are five digits, on these airports there are only small number of flights.

In [12]:
flights_delays_reduced.filter((pl.col('DESTINATION_AIRPORT').cast(pl.String).str.len_chars() == 5) | (
            pl.col('DESTINATION_AIRPORT').cast(pl.String).str.len_chars() == 5)).collect()

YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_DELAY
i32,i32,i32,i32,cat,cat,cat,cat,time,f64,f64,i64,time,f64
2015,10,1,4,"""AA""","""N3DBAA""","""14747""","""11298""",00:05:00,10.0,237.0,1660,06:02:00,-19.0
2015,10,1,4,"""DL""","""N696DL""","""14771""","""13487""",00:05:00,11.0,213.0,1589,05:38:00,-10.0
2015,10,1,4,"""NK""","""N602NK""","""12889""","""13487""",00:05:00,-5.0,177.0,1299,05:02:00,-14.0
2015,10,1,4,"""AA""","""N3GNAA""","""12892""","""13303""",00:10:00,-3.0,296.0,2342,08:06:00,7.0
2015,10,1,4,"""AA""","""N914UY""","""14771""","""11057""",00:10:00,-2.0,291.0,2296,08:01:00,-11.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2015,10,31,6,"""B6""","""N715JB""","""11292""","""12478""",23:59:00,15.0,208.0,1626,04:27:00,0.0
2015,10,31,6,"""B6""","""N834JB""","""12478""","""10732""",23:59:00,-1.0,220.0,1576,03:39:00,-17.0
2015,10,31,6,"""B6""","""N907JB""","""12478""","""14843""",23:59:00,93.0,223.0,1598,03:42:00,80.0


There are 482,878 rows with airport that don't have standard IATA code, so they will be dropped.

In [13]:
flights_delays_reduced = flights_delays_reduced.filter(
    (pl.col('DESTINATION_AIRPORT').cast(pl.String).str.len_chars() != 5) & (
                pl.col('DESTINATION_AIRPORT').cast(pl.String).str.len_chars() != 5))

## Feature engineering

Basic transformations
- SEASON – based on MONTH
- IS_WEEKEND – based on DAY_OF_WEEK
- IS_HOLIDAY – checked with "holidays" library
- DEP_HOUR – hour based on SCHEDULED_DEPARTURE
- DEP_MINUTE – minute based on SCHEDULED_DEPARTURE
- ARR_HOUR – hour based on SCHEDULED_ARRIVAL
- ARR_MINUTE – minute based on SCHEDULED_ARRIVAL

In [14]:
def get_season(month):
    if 3 <= month < 6:
        return "SPRING"
    elif 6 <= month < 9:
        return "SUMMER"
    elif 9 <= month < 12:
        return "AUTUMN"
    else:
        return "WINTER"


import holidays

holidays = holidays.US(years=2015)

flights_delays_reduced_features1 = (flights_delays_reduced.
with_columns(
    SEASON=pl.col('MONTH').cast(pl.String).cast(pl.Int32).map_elements(get_season, return_dtype=pl.String),
    IS_WEEKEND=pl.col('DAY_OF_WEEK').cast(pl.String).cast(pl.Int32).is_in([6, 7]).cast(pl.Int32),
    IS_HOLIDAY=pl.concat_str(["YEAR", "MONTH", "DAY"], separator="-").str.to_date("%Y-%m-%d").is_in(holidays).cast(
        pl.Int32),
    DEP_HOUR=pl.col('SCHEDULED_DEPARTURE').dt.hour(),
    DEP_MINUTE=pl.col('SCHEDULED_DEPARTURE').dt.minute(),
    ARR_HOUR=pl.col('SCHEDULED_ARRIVAL').dt.hour(),
    ARR_MINUTE=pl.col('SCHEDULED_ARRIVAL').dt.minute(),
))

flights_delays_reduced_features1.collect()

YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_DELAY,SEASON,IS_WEEKEND,IS_HOLIDAY,DEP_HOUR,DEP_MINUTE,ARR_HOUR,ARR_MINUTE
i32,i32,i32,i32,cat,cat,cat,cat,time,f64,f64,i64,time,f64,str,i32,i32,i8,i8,i8,i8
2015,1,1,4,"""AS""","""N407AS""","""ANC""","""SEA""",00:05:00,-11.0,205.0,1448,04:30:00,-22.0,"""WINTER""",0,1,0,5,4,30
2015,1,1,4,"""AA""","""N3KUAA""","""LAX""","""PBI""",00:10:00,-8.0,280.0,2330,07:50:00,-9.0,"""WINTER""",0,1,0,10,7,50
2015,1,1,4,"""US""","""N171US""","""SFO""","""CLT""",00:20:00,-2.0,286.0,2296,08:06:00,5.0,"""WINTER""",0,1,0,20,8,6
2015,1,1,4,"""AA""","""N3HYAA""","""LAX""","""MIA""",00:20:00,-5.0,285.0,2342,08:05:00,-9.0,"""WINTER""",0,1,0,20,8,5
2015,1,1,4,"""AS""","""N527AS""","""SEA""","""ANC""",00:25:00,-1.0,235.0,1448,03:20:00,-21.0,"""WINTER""",0,1,0,25,3,20
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2015,12,31,4,"""B6""","""N657JB""","""LAX""","""BOS""",23:59:00,-4.0,320.0,2611,08:19:00,-26.0,"""WINTER""",0,0,23,59,8,19
2015,12,31,4,"""B6""","""N828JB""","""JFK""","""PSE""",23:59:00,-4.0,227.0,1617,04:46:00,-16.0,"""WINTER""",0,0,23,59,4,46
2015,12,31,4,"""B6""","""N913JB""","""JFK""","""SJU""",23:59:00,-9.0,221.0,1598,04:40:00,-8.0,"""WINTER""",0,0,23,59,4,40


More complex transformations with statistics
- Actual state of airplane and airport
    - PREVIOUS_AIRPLANE_DELAY – Checks the airplane's previous flight delay
    - PREVIOUS_FLIGHT_ON_AIRPORT_DELAY – Checks the latest flight on a certain airport delay
    - AMT_PLANES_ON_AIRPORT – Calculates how many flights depart at this hour on a certain day
- Statistics about previous flights
     - per Airport
        - MAX_AIRPORT_DEPT_DELAY – Max of airport departure delays
        - MIN_AIRPORT_DEPT_DELAY – Min of airport departure delays
        - AVG_AIRPORT_DEPT_DELAY – Mean of airport departure delays
        - MEDIAN_AIRPORT_DEPT_DELAY – Median of airport departure delays
        - RATE_AIRPORT_DEPT_DELAY_0 – Ratio of delayed by 0 minutes or more on departure to all flights from the airport
        - RATE_AIRPORT_DEPT_DELAY_15 – Ratio of delayed by 15 minutes or more on departure to all flights from the airport
        - RATE_AIRPORT_DEPT_DELAY_30 – Ratio of delayed by 30 minutes or more on departure to all flights from the airport

        - MAX_AIRPORT_ARR_DELAY – Max of airport arrival delays
        - MIN_AIRPORT_ARR_DELAY – Min of airport arrival delays
        - AVG_AIRPORT_ARR_DELAY – Mean of airport arrival delays
        - MEDIAN_AIRPORT_ARR_DELAY – Median of airport arrival delays
        - RATE_AIRPORT_ARR_DELAY_0 – Ratio of delayed by 0 minutes or more on arrival to all flights from the airport
        - RATE_AIRPORT_ARR_DELAY_15 – Ratio of delayed by 15 minutes or more on arrival to all flights from the airport
        - RATE_AIRPORT_ARR_DELAY_30 – Ratio of delayed by 30 minutes or more on arrival to all flights from the airport

        - AIRPORT_TOTAL_FLIGHTS – Number of flights from this airport

     - per Airline
        - MAX_AIRLINE_DEPT_DELAY – Max of airline departure delays
        - MIN_AIRLINE_DEPT_DELAY – Min of airline departure delays
        - AVG_AIRLINE_DEPT_DELAY – Mean of airline departure delays
        - MEDIAN_AIRLINE_DEPT_DELAY – Median of airline departure delays
        - RATE_AIRLINE_DEPT_DELAY_0 – Ratio of delayed by 0 minutes or more on departure to all flights by this airline
        - RATE_AIRLINE_DEPT_DELAY_15 – Ratio of delayed by 15 minutes or more on departure to all flights by this airline
        - RATE_AIRLINE_DEPT_DELAY_30 – Ratio of delayed by 30 minutes or more on departure to all flights by this airline

        - MAX_AIRLINE_ARR_DELAY – Max of airline arrival delays
        - MIN_AIRLINE_ARR_DELAY – Min of airline arrival delays
        - AVG_AIRLINE_ARR_DELAY – Mean of airline arrival delays
        - MEDIAN_AIRLINE_ARR_DELAY – Median of airline arrival delays
        - RATE_AIRLINE_ARR_DELAY_0 – Ratio of delayed by 0 minutes or more on arrival to all flights by this airline
        - RATE_AIRLINE_ARR_DELAY_15 – Ratio of delayed by 15 minutes or more on arrival to all flights by this airline
        - RATE_AIRLINE_ARR_DELAY_30 – Ratio of delayed by 30 minutes or more on arrival to all flights by this airline

        - AIRLINE_TOTAL_FLIGHTS – Number of flights by this airline

     - per Route
        - MAX_ROUTE_DEPT_DELAY – Max of route departure delays
        - MIN_ROUTE_DEPT_DELAY – Min of route departure delays
        - AVG_ROUTE_DEPT_DELAY – Mean of route departure delays
        - MEDIAN_ROUTE_DEPT_DELAY – Median of route departure delays
        - RATE_ROUTE_DEPT_DELAY_0 – Ratio of delayed by 0 minutes or more on departure to all flights on this route
        - RATE_ROUTE_DEPT_DELAY_15 – Ratio of delayed by 15 minutes or more on departure to all flights on this route
        - RATE_ROUTE_DEPT_DELAY_30 – Ratio of delayed by 30 minutes or more on departure to all flights on this route

        - MAX_ROUTE_ARR_DELAY – Max of route arrival delays
        - MIN_ROUTE_ARR_DELAY – Min of route arrival delays
        - AVG_ROUTE_ARR_DELAY – Mean of route arrival delays
        - MEDIAN_ROUTE_ARR_DELAY – Median of route arrival delays
        - RATE_ROUTE_ARR_DELAY_0 – Ratio of delayed by 0 minutes or more on arrival to all flights on this route
        - RATE_ROUTE_ARR_DELAY_15 – Ratio of delayed by 15 minutes or more on arrival to all flights on this route
        - RATE_ROUTE_ARR_DELAY_30 – Ratio of delayed by 30 minutes or more on arrival to all flights on this route

        - AIRPORT_TOTAL_FLIGHTS – Number of flights on this route

     - per HOUR
        - MAX_HOUR_DEPT_DELAY – Max of hour departure delays
        - MIN_HOUR_DEPT_DELAY – Min of hour departure delays
        - AVG_HOUR_DEPT_DELAY – Mean of hour departure delays
        - MEDIAN_HOUR_DEPT_DELAY – Median of hour departure delays
        - RATE_HOUR_DEPT_DELAY_0 – Ratio of delayed by 0 minutes or more on departure to all flights at this hour
        - RATE_HOUR_DEPT_DELAY_15 – Ratio of delayed by 15 minutes or more on departure to all flights at this hour
        - RATE_HOUR_DEPT_DELAY_30 – Ratio of delayed by 30 minutes or more on departure to all flights at this hour

        - MAX_HOUR_ARR_DELAY – Max of hour arrival delays
        - MIN_HOUR_ARR_DELAY – Min of hour arrival delays
        - AVG_HOUR_ARR_DELAY – Mean of hour arrival delays
        - MEDIAN_HOUR_ARR_DELAY – Median of hour arrival delays
        - RATE_HOUR_ARR_DELAY_0 – Ratio of delayed by 0 minutes or more on arrival to all flights at this hour
        - RATE_HOUR_ARR_DELAY_15 – Ratio of delayed by 15 minutes or more on arrival to all flights at this hour
        - RATE_HOUR_ARR_DELAY_30 – Ratio of delayed by 30 minutes or more on arrival to all flights at this hour


In [15]:
import polars as pl


def delay_aggs(delay_col: str, prefix: str) -> list:
    return [
        pl.col(delay_col).max().alias(f"MAX_{prefix}"),
        pl.col(delay_col).min().alias(f"MIN_{prefix}"),
        pl.col(delay_col).mean().alias(f"AVG_{prefix}"),
        pl.col(delay_col).median().alias(f"MEDIAN_{prefix}"),
        (pl.col(delay_col) >= 0).mean().alias(f"RATE_{prefix}_0"),
        (pl.col(delay_col) >= 15).mean().alias(f"RATE_{prefix}_15"),
        (pl.col(delay_col) >= 30).mean().alias(f"RATE_{prefix}_30"),
    ]


# per AIRPORT
# ============================================================
airport_stats = (
    flights_delays_reduced_features1
    .group_by("ORIGIN_AIRPORT")
    .agg(
        *delay_aggs("DEPARTURE_DELAY", "AIRPORT_DEPT_DELAY"),
        *delay_aggs("ARRIVAL_DELAY", "AIRPORT_ARR_DELAY"),
        pl.len().alias("AIRPORT_TOTAL_FLIGHTS"),
    )
)

# Statystyki per AIRLINE
# ============================================================
airline_stats = (
    flights_delays_reduced_features1
    .group_by("AIRLINE")
    .agg(
        *delay_aggs("DEPARTURE_DELAY", "AIRLINE_DEPT_DELAY"),
        *delay_aggs("ARRIVAL_DELAY", "AIRLINE_ARR_DELAY"),
        pl.len().alias("AIRLINE_TOTAL_FLIGHTS"),
    )
)

# Statystyki per ROUTE (ORIGIN + DESTINATION + AIRLINE)
# ============================================================
route_stats = (
    flights_delays_reduced_features1
    .group_by(["ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "AIRLINE"])
    .agg(
        *delay_aggs("DEPARTURE_DELAY", "ROUTE_DEPT_DELAY"),
        *delay_aggs("ARRIVAL_DELAY", "ROUTE_ARR_DELAY"),
        pl.len().alias("ROUTE_TOTAL_FLIGHTS"),
    )
)

# Statystyki per HOUR
# ============================================================
hour_stats = (
    flights_delays_reduced_features1
    .group_by("DEP_HOUR")
    .agg(
        *delay_aggs("DEPARTURE_DELAY", "HOUR_DEPT_DELAY"),
        *delay_aggs("ARRIVAL_DELAY", "HOUR_ARR_DELAY"),
    )
)

# PREVIOUS_AIRPLANE_DELAY
# ============================================================
flights_with_prev = (
    flights_delays_reduced_features1
    .sort(["TAIL_NUMBER", "YEAR", "MONTH", "DAY", "SCHEDULED_DEPARTURE"])
    .with_columns(
        pl.col("ARRIVAL_DELAY")
        .shift(1)
        .over("TAIL_NUMBER")
        .alias("PREVIOUS_AIRPLANE_DELAY")
    )
)

# PREVIOUS_FLIGHT_ON_AIRPORT_DELAY
# ============================================================
flights_with_prev = (
    flights_with_prev
    .sort(["ORIGIN_AIRPORT", "YEAR", "MONTH", "DAY", "SCHEDULED_DEPARTURE"])
    .with_columns(
        pl.col("DEPARTURE_DELAY")
        .shift(1)
        .over("ORIGIN_AIRPORT")
        .alias("PREVIOUS_FLIGHT_ON_AIRPORT_DELAY")
    )
)

# AMT_PLANES_ON_AIRPORT
# ============================================================
slot_counts = (
    flights_delays_reduced_features1
    .group_by(["ORIGIN_AIRPORT", "YEAR", "MONTH", "DAY", "DEP_HOUR"])
    .agg(pl.len().alias("AMT_PLANES_ON_AIRPORT"))
)

# JOIN wszystkiego do głównej tabeli
# ============================================================
flights_features = (
    flights_with_prev
    .join(airport_stats, on="ORIGIN_AIRPORT", how="left")
    .join(airline_stats, on="AIRLINE", how="left")
    .join(route_stats, on=["ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "AIRLINE"], how="left")
    .join(hour_stats, on="DEP_HOUR", how="left")
    .join(slot_counts, on=["ORIGIN_AIRPORT", "YEAR", "MONTH", "DAY", "DEP_HOUR"], how="left")
)

print(flights_features.collect_schema().names())

['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_DELAY', 'SCHEDULED_TIME', 'DISTANCE', 'SCHEDULED_ARRIVAL', 'ARRIVAL_DELAY', 'SEASON', 'IS_WEEKEND', 'IS_HOLIDAY', 'DEP_HOUR', 'DEP_MINUTE', 'ARR_HOUR', 'ARR_MINUTE', 'PREVIOUS_AIRPLANE_DELAY', 'PREVIOUS_FLIGHT_ON_AIRPORT_DELAY', 'MAX_AIRPORT_DEPT_DELAY', 'MIN_AIRPORT_DEPT_DELAY', 'AVG_AIRPORT_DEPT_DELAY', 'MEDIAN_AIRPORT_DEPT_DELAY', 'RATE_AIRPORT_DEPT_DELAY_0', 'RATE_AIRPORT_DEPT_DELAY_15', 'RATE_AIRPORT_DEPT_DELAY_30', 'MAX_AIRPORT_ARR_DELAY', 'MIN_AIRPORT_ARR_DELAY', 'AVG_AIRPORT_ARR_DELAY', 'MEDIAN_AIRPORT_ARR_DELAY', 'RATE_AIRPORT_ARR_DELAY_0', 'RATE_AIRPORT_ARR_DELAY_15', 'RATE_AIRPORT_ARR_DELAY_30', 'AIRPORT_TOTAL_FLIGHTS', 'MAX_AIRLINE_DEPT_DELAY', 'MIN_AIRLINE_DEPT_DELAY', 'AVG_AIRLINE_DEPT_DELAY', 'MEDIAN_AIRLINE_DEPT_DELAY', 'RATE_AIRLINE_DEPT_DELAY_0', 'RATE_AIRLINE_DEPT_DELAY_15', 'RATE_AIRLINE_DEPT_DELAY_30', 'MAX_AIRLINE_ARR_DELAY',

Repeated null check after feature engineering

In [18]:
(flights_features
 .null_count()
 .unpivot()
 .filter(pl.col('value') > 0)
 .collect())

variable,value
str,u32
"""PREVIOUS_AIRPLANE_DELAY""",4895
"""PREVIOUS_FLIGHT_ON_AIRPORT_DEL…",322


There are two situations:

    - There is no previous flight of certain plane in this data, e.g. null is in the row of the first flight in this year for airplane
    - There is no previous flight on certain airport, e.g. null is in the row of the first flight in this year on airport

In both situation we can assume that there is no previous delay because we don't know data from the previous year and we will fill these nulls with zeros.

In [24]:
flights_features=flights_features.with_columns(
    pl.col("PREVIOUS_AIRPLANE_DELAY").fill_null(0),
                              pl.col("PREVIOUS_FLIGHT_ON_AIRPORT_DELAY").fill_null(0)
)

As it was mentioned YEAR, DEPARTURE_DELAY and TAIL_NUMBER should be dropped

In [17]:
flights_features=flights_features.drop(["YEAR", "DEPARTURE_DELAY", "TAIL_NUMBER","SCHEDULED_DEPARTURE","SCHEDULED_ARRIVAL"])
print(flights_features.collect_schema().names())


['MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_TIME', 'DISTANCE', 'ARRIVAL_DELAY', 'SEASON', 'IS_WEEKEND', 'IS_HOLIDAY', 'DEP_HOUR', 'DEP_MINUTE', 'ARR_HOUR', 'ARR_MINUTE', 'PREVIOUS_AIRPLANE_DELAY', 'PREVIOUS_FLIGHT_ON_AIRPORT_DELAY', 'MAX_AIRPORT_DEPT_DELAY', 'MIN_AIRPORT_DEPT_DELAY', 'AVG_AIRPORT_DEPT_DELAY', 'MEDIAN_AIRPORT_DEPT_DELAY', 'RATE_AIRPORT_DEPT_DELAY_0', 'RATE_AIRPORT_DEPT_DELAY_15', 'RATE_AIRPORT_DEPT_DELAY_30', 'MAX_AIRPORT_ARR_DELAY', 'MIN_AIRPORT_ARR_DELAY', 'AVG_AIRPORT_ARR_DELAY', 'MEDIAN_AIRPORT_ARR_DELAY', 'RATE_AIRPORT_ARR_DELAY_0', 'RATE_AIRPORT_ARR_DELAY_15', 'RATE_AIRPORT_ARR_DELAY_30', 'AIRPORT_TOTAL_FLIGHTS', 'MAX_AIRLINE_DEPT_DELAY', 'MIN_AIRLINE_DEPT_DELAY', 'AVG_AIRLINE_DEPT_DELAY', 'MEDIAN_AIRLINE_DEPT_DELAY', 'RATE_AIRLINE_DEPT_DELAY_0', 'RATE_AIRLINE_DEPT_DELAY_15', 'RATE_AIRLINE_DEPT_DELAY_30', 'MAX_AIRLINE_ARR_DELAY', 'MIN_AIRLINE_ARR_DELAY', 'AVG_AIRLINE_ARR_DELAY', 'MEDIAN_AIRLINE_ARR_DELAY', 'RATE_A

## Feature importance test with Mutual Information

In [28]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import OrdinalEncoder
import polars.selectors as cs
import numpy as np


data_frame_pandas=flights_features.collect().to_pandas()

feature_names=flights_features.collect_schema().names()
feature_names.remove('ARRIVAL_DELAY')

X=data_frame_pandas[feature_names]
y=data_frame_pandas['ARRIVAL_DELAY']

cat_cols = (
    flights_features
    .select(cs.by_dtype([pl.Categorical, pl.String]))
    .collect_schema()
    .names()
)

enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = enc.fit_transform(X[cat_cols].astype(str))

mi = mutual_info_regression(X, y, random_state=42)

mi_df = (
    pl.DataFrame({"feature": feature_names, "mutual_info": mi})
    .sort("mutual_info", descending=True)
)

MemoryError: Unable to allocate 39.9 MiB for an array with shape (5231130,) and data type float64

In [ ]:
import polars.selectors as cs
feature_names=flights_features.collect_schema().names()
feature_names.remove('ARRIVAL_DELAY')

categorical_features=flights_features.select(cs.by_dtype([pl.Categorical,pl.String])).collect_schema().names()

feature_names = [name for name in feature_names if name not in categorical_features]

print(feature_names)